# RAG: Retrieval-Augmented Generation (Lewis et al., 2020)

A language model stores what it knows in its weights. The paper calls this
**parametric memory**. It has three problems. It cannot be updated without
retraining. It cannot cite where an answer came from. And when it does not
know, it guesses fluently.

*Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks* adds a
**non-parametric memory**, a searchable index of documents, and lets the model
read from it before answering:

1. A **retriever** (DPR, a BERT bi-encoder) embeds the question and fetches the
   top-$k$ passages from an index of 21 million 100-word Wikipedia chunks.
2. A **generator** (BART-large) reads question + passage and writes the answer.
3. The two are trained together, end to end, with the retrieved passage treated
   as a **latent variable** that is marginalised out.

This page builds the retrieval step by hand, works through the two ways the
paper marginalises over passages, measures where simple retrieval fails, and
shows why the index can be swapped without retraining anything.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

from hmw_motion import Scene, mix

np.set_printoptions(precision=4, suppress=True, linewidth=120)


def banner(title):
    print("=" * 78); print(f"  {title}"); print("=" * 78)


def step(label):
    print(); print(label); print("-" * 78)


def check(label, ok):
    print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return ok


print("helpers ready")

helpers ready


## 1. An index to retrieve from

The real system indexes Wikipedia. Here the index is 16 short passages, about
the papers in this series and a few distractors, so every score can be printed.

In [2]:
docs = {
    "transformer": "The Transformer architecture relies entirely on attention and drops recurrence. "
                   "It was introduced for machine translation in 2017.",
    "gpt": "GPT pre-trains a decoder on unlabelled books by predicting the next token, then fine-tunes "
           "it on each labelled task.",
    "bert": "BERT masks fifteen percent of input tokens and trains a bidirectional encoder to recover them.",
    "lora": "LoRA freezes the pre-trained weights and trains a low-rank update made of two small matrices.",
    "peft": "Adapters, prefix tuning and prompt tuning adapt large models by training only a few new parameters.",
    "rag": "Retrieval-augmented generation fetches passages from an index and conditions the generator on them.",
    "vae": "A variational autoencoder maximises the evidence lower bound using the reparameterisation trick.",
    "gan": "A generative adversarial network trains a generator against a discriminator in a minimax game.",
    "diffusion": "Diffusion models add Gaussian noise step by step and learn to reverse the corruption.",
    "vit": "The Vision Transformer splits an image into sixteen by sixteen patches and treats them as tokens.",
    "rope": "Rotary position embedding rotates queries and keys so attention depends on relative position.",
    "helsinki": "Helsinki is the capital of Finland and lies on the coast of the Gulf of Finland.",
    "coffee": "Coffee beans are roasted seeds of the coffee plant, first cultivated in Yemen.",
    "football": "A football match lasts ninety minutes, played as two halves of forty-five minutes.",
    "python": "Python is a programming language that uses indentation to delimit blocks of code.",
    "everest": "Mount Everest is the highest mountain above sea level, on the border of Nepal and China.",
}
ids = list(docs)
vec = TfidfVectorizer(stop_words="english")
Dm = vec.fit_transform([docs[i] for i in ids])
print(f"  {len(ids)} passages, vocabulary of {len(vec.vocabulary_)} terms after removing stop words")

  16 passages, vocabulary of 132 terms after removing stop words


## 2. Retrieval as a nearest-neighbour search

The paper's retriever scores a passage $z$ for a question $x$ with an inner
product of two embeddings:

$$p_\eta(z \mid x) \propto \exp\big(\mathbf{d}(z)^\top \mathbf{q}(x)\big)$$

$\mathbf{d}$ and $\mathbf{q}$ are two BERT encoders. Because the score is a plain
inner product, all 21M passage vectors can be computed once and searched with
maximum inner product search (FAISS) in milliseconds.

The same structure works with a much simpler encoder: TF-IDF vectors, which
are L2-normalised so the inner product is the cosine similarity. The cell below
computes one score by hand and then ranks the whole index.

In [3]:
banner("RETRIEVE FOR ONE QUESTION")
question = "Which method trains a low-rank update instead of all the weights?"
q = vec.transform([question])
scores = (Dm @ q.T).toarray().ravel()

step("Terms shared by the question and the 'lora' passage")
vocab_inv = {i: t for t, i in vec.vocabulary_.items()}
qv, dv = q.toarray().ravel(), Dm[ids.index("lora")].toarray().ravel()
shared = np.nonzero(qv * dv)[0]
tot = 0.0
for j in shared:
    tot += qv[j] * dv[j]
    print(f"  {vocab_inv[j]:<10} q = {qv[j]:.4f}   d = {dv[j]:.4f}   product = {qv[j]*dv[j]:.4f}")
print(f"  sum = {tot:.4f}")
check("by-hand sum == sparse inner product", np.isclose(tot, scores[ids.index("lora")]))

step("Ranking the whole index")
order = np.argsort(-scores)
for r, j in enumerate(order[:5], 1):
    print(f"  {r}. {ids[j]:<11} score {scores[j]:.4f}   {docs[ids[j]][:62]}...")
check("top-1 is the LoRA passage", ids[order[0]] == "lora")

  RETRIEVE FOR ONE QUESTION

Terms shared by the question and the 'lora' passage
------------------------------------------------------------------------------
  low        q = 0.4713   d = 0.3122   product = 0.1471
  rank       q = 0.4713   d = 0.3122   product = 0.1471
  trains     q = 0.3338   d = 0.2211   product = 0.0738
  update     q = 0.4713   d = 0.3122   product = 0.1471
  weights    q = 0.4713   d = 0.3122   product = 0.1471
  sum = 0.6624
  [PASS] by-hand sum == sparse inner product

Ranking the whole index
------------------------------------------------------------------------------
  1. lora        score 0.6624   LoRA freezes the pre-trained weights and trains a low-rank upd...
  2. gan         score 0.0877   A generative adversarial network trains a generator against a ...
  3. bert        score 0.0822   BERT masks fifteen percent of input tokens and trains a bidire...
  4. gpt         score 0.0704   GPT pre-trains a decoder on unlabelled books by predicting the...
  5.

## 3. Measuring retrieval: recall@k

The generator can only use what the retriever hands it, so retrieval quality
caps the whole system. The standard measure is **recall@k**: the fraction of
questions whose gold passage is among the top $k$.

Ten questions with known answers. The last four are deliberately
**rephrased**, sharing few or no content words with their passage. That is the case where
term-matching fails, and it is why the paper uses a learned dense retriever.

In [4]:
banner("RECALL@k OVER TEN QUESTIONS")
qs = [
    ("What does BERT mask during pre-training?", "bert"),
    ("How do diffusion models corrupt their training data?", "diffusion"),
    ("Which model splits an image into patches?", "vit"),
    ("What is the capital of Finland?", "helsinki"),
    ("How long does a football match last?", "football"),
    ("Which network plays a minimax game between two players?", "gan"),
    ("Where is the tallest peak on Earth?", "everest"),                  # paraphrase
    ("Which seeds are roasted for a morning drink?", "coffee"),          # partial overlap
    ("How can I adapt a huge network cheaply?", "peft"),                 # paraphrase
    ("How does a model know the order of words?", "rope"),               # paraphrase
]
ranks = []
for qq, gold in qs:
    sc_ = (Dm @ vec.transform([qq]).T).toarray().ravel()
    rk = int(np.where(np.argsort(-sc_) == ids.index(gold))[0][0]) + 1
    ranks.append(rk)
    flag = "" if rk == 1 else "   <- missed at k=1" if sc_.max() > 0 else "   <- zero overlap"
    print(f"  rank {rk:>2}  {gold:<10} {qq}{flag}")
ranks = np.array(ranks)
for k in [1, 3, 5]:
    print(f"  recall@{k} = {(ranks <= k).mean():.2f}")

  RECALL@k OVER TEN QUESTIONS
  rank  1  bert       What does BERT mask during pre-training?
  rank  1  diffusion  How do diffusion models corrupt their training data?
  rank  1  vit        Which model splits an image into patches?
  rank  1  helsinki   What is the capital of Finland?
  rank  1  football   How long does a football match last?
  rank  1  gan        Which network plays a minimax game between two players?
  rank 16  everest    Where is the tallest peak on Earth?   <- zero overlap
  rank  1  coffee     Which seeds are roasted for a morning drink?
  rank  2  peft       How can I adapt a huge network cheaply?   <- missed at k=1
  rank 11  rope       How does a model know the order of words?   <- zero overlap
  recall@1 = 0.70
  recall@3 = 0.80
  recall@5 = 0.80


Questions that reuse the passage's words are found at rank 1. *"Where is the
tallest peak on Earth?"* shares no content word with *"Mount Everest is the
highest mountain above sea level"*, so its TF-IDF score is exactly zero. A
dense retriever trained on question–answer pairs learns that *tallest peak* and
*highest mountain* mean the same thing. That is the job of DPR (Karpukhin et
al., 2020), which RAG uses and fine-tunes.

The animation lays the passages out in 2-D so that their distances roughly
follow the cosine distances. The ranking itself is computed in the full
132-dimensional space.

In [5]:
from sklearn.manifold import MDS

# 2-D layout that tries to preserve the cosine distances (passages + question)
allv = np.vstack([Dm.toarray(), q.toarray()])
dist = 1 - allv @ allv.T
np.fill_diagonal(dist, 0)
emb2 = MDS(2, dissimilarity="precomputed", random_state=1, n_init=8).fit_transform(np.clip(dist, 0, None))
P2, qp = emb2[:-1], emb2[-1]
lo, hi = emb2.min(0), emb2.max(0)


def to_px(p):
    return 50 + (p[0] - lo[0]) / (hi[0] - lo[0]) * 340, 345 - (p[1] - lo[1]) / (hi[1] - lo[1]) * 270


def retrieval_scene():
    sc = Scene(720, 400, dur=11.0, caption="Retrieve, then generate: the question's top-3 passages become the context",
               poster=10.0)
    sc.rect(20, 20, 440, 350, fill="panel", rx=10)
    sc.text(240, 38, "passages laid out by cosine distance (MDS, 2-D)", size=11, fill="muted")
    dots = {}
    for i, d_ in enumerate(ids):
        x, y = to_px(P2[i])
        dots[d_] = sc.circle(x, y, r=6, fill="violet", opacity=0)
        dots[d_].to(0.1 + 0.04 * i, 0.5 + 0.04 * i, opacity=0.9)
        sc.text(x + 9, y - 9, d_, size=10, fill="muted", anchor="start", opacity=0).to(0.1 + 0.04 * i, 0.5 + 0.04 * i,
                                                                                    opacity=1)
    qx, qy = to_px(qp)
    qd = sc.circle(qx, qy, r=0, fill="amber")
    qd.to(1.6, 2.1, r=9, ease="back")
    sc.text(qx, qy + 22, "question", size=11, fill="amber", weight=700, opacity=0).fade_in(1.8)
    sc.text(590, 40, "context for the generator", size=12, fill="muted")
    top3 = [ids[j] for j in order[:3]]
    for r, d_ in enumerate(top3):
        x, y = to_px(P2[ids.index(d_)])
        ln = sc.line(qx, qy, qx, qy, stroke="amber", sw=2, opacity=0.8)
        ln.to(2.6 + 0.5 * r, 3.1 + 0.5 * r, x2=x, y2=y)
        dots[d_].to(2.6 + 0.5 * r, 3.1 + 0.5 * r, r=9, fill="cyan")
        card = sc.rect(x, y, 0, 0, fill=mix("panel", "cyan", 0.25), stroke="cyan", sw=1.2, rx=6, opacity=0)
        card.to(4.6 + 0.4 * r, 5.4 + 0.4 * r, x=475, y=60 + r * 62, w=230, h=52, opacity=1)
        sc.text(485, 76 + r * 62, f"#{r+1}  {d_}   score {scores[ids.index(d_)]:.3f}", size=11, fill="cyan",
                anchor="start", weight=700, opacity=0).fade_in(5.3 + 0.4 * r)
        sc.text(485, 96 + r * 62, docs[d_][:38] + "...", size=10, fill="text", anchor="start",
                opacity=0).fade_in(5.3 + 0.4 * r)
    sc.rect(475, 268, 230, 44, fill="bg", stroke="green", sw=1.5, rx=6, opacity=0).fade_in(7.2)
    sc.text(590, 290, "generator (BART) reads question + passage", size=10, fill="green", opacity=0).fade_in(7.2)
    sc.text(590, 340, "answer: LoRA", size=16, fill="white", weight=700, opacity=0).to(8.2, 8.8, opacity=1, y=334)
    sc.beat(0.0, "Every passage is embedded once, ahead of time, into the index.")
    sc.beat(1.6, "The question is embedded with the query encoder.")
    sc.beat(2.6, "Nearest neighbours by inner product: the top-k passages.")
    sc.beat(4.6, "They are handed to the generator with the question...")
    sc.beat(7.2, "...which writes the answer, grounded in text it can point to.")
    return sc

retrieval_scene()

<motion scene: Retrieve, then generate: the question's top-3 passages become the context (52 elements, 11s)>

## 4. Two ways to use $k$ passages

Retrieval returns $k$ passages, and the paper treats "which passage" as a
hidden variable to be summed out. It defines two models.

**RAG-Sequence** picks one passage for the whole answer, then averages over
passages:

$$p(y \mid x) = \sum_{z \in \text{top-}k} p_\eta(z \mid x) \prod_{i} p_\theta(y_i \mid x, z, y_{<i})$$

**RAG-Token** may switch passage at every token:

$$p(y \mid x) = \prod_{i} \sum_{z \in \text{top-}k} p_\eta(z \mid x)\, p_\theta(y_i \mid x, z, y_{<i})$$

The difference is whether the sum sits outside or inside the product. RAG-Token
can assemble an answer from facts spread across several passages.

A worked example with 3 passages, a 3-word vocabulary and answers 2 tokens
long. That is small enough to enumerate all 9 answers and check that each model
is a proper probability distribution.

In [6]:
banner("RAG-SEQUENCE vs RAG-TOKEN, ENUMERATED")
pz = np.array([0.6, 0.3, 0.1])                      # retriever: p(z | x)
V = ["Paris", "France", "Lyon"]
# generator p(y_i | x, z): each passage favours different tokens at each position
gen = np.array([
    [[0.80, 0.15, 0.05], [0.10, 0.80, 0.10]],        # passage 1: "Paris France"
    [[0.20, 0.10, 0.70], [0.15, 0.75, 0.10]],        # passage 2: "Lyon France"
    [[0.34, 0.33, 0.33], [0.34, 0.33, 0.33]],        # passage 3: uninformative
])
seq, tok = {}, {}
for a in range(3):
    for b in range(3):
        seq[(a, b)] = sum(pz[z] * gen[z, 0, a] * gen[z, 1, b] for z in range(3))
        tok[(a, b)] = (pz @ gen[:, 0, a]) * (pz @ gen[:, 1, b])
print(f"  {'answer':<16} {'RAG-Sequence':>13} {'RAG-Token':>11}")
for (a, b) in sorted(seq, key=lambda k: -seq[k]):
    print(f"  {V[a] + ' ' + V[b]:<16} {seq[(a, b)]:>13.4f} {tok[(a, b)]:>11.4f}")
step("Checks")
check("RAG-Sequence probabilities sum to 1", np.isclose(sum(seq.values()), 1))
check("RAG-Token probabilities sum to 1", np.isclose(sum(tok.values()), 1))
a_, b_ = 0, 1
by_hand = 0.6 * 0.80 * 0.80 + 0.3 * 0.20 * 0.75 + 0.1 * 0.34 * 0.33
print(f"\n  p_seq(Paris France) by hand = 0.6*0.80*0.80 + 0.3*0.20*0.75 + 0.1*0.34*0.33 = {by_hand:.4f}")
check("matches the table", np.isclose(by_hand, seq[(a_, b_)]))

  RAG-SEQUENCE vs RAG-TOKEN, ENUMERATED
  answer            RAG-Sequence   RAG-Token
  Paris France            0.4402      0.4236
  Lyon France             0.1924      0.2015
  France France           0.1054      0.1129
  Paris Paris             0.0686      0.0798
  Paris Lyon              0.0652      0.0706
  Lyon Paris              0.0457      0.0379
  Lyon Lyon               0.0349      0.0336
  France Paris            0.0247      0.0213
  France Lyon             0.0229      0.0188

Checks
------------------------------------------------------------------------------
  [PASS] RAG-Sequence probabilities sum to 1
  [PASS] RAG-Token probabilities sum to 1

  p_seq(Paris France) by hand = 0.6*0.80*0.80 + 0.3*0.20*0.75 + 0.1*0.34*0.33 = 0.4402
  [PASS] matches the table


Both distributions are valid, and they disagree. The sequence model rewards
answers that a single passage supports from start to finish: *Paris France*
gets 0.440. The token model mixes passages at every position, so probability
moves toward answers stitched together from different passages. *Paris Paris*
and *Paris Lyon* both rise, and *Paris France* drops to 0.424. In the paper,
RAG-Token did better on tasks that combine facts from several passages, and
RAG-Sequence on some open-domain QA.

In [7]:
def marg_scene():
    sc = Scene(720, 380, dur=10.0, caption="RAG-Sequence: weight each passage's answer distribution by p(z|x), then add",
               poster=9.5)
    cols = ["cyan", "pink", "muted"]
    answers = [(0, 1), (2, 1), (0, 0), (2, 0)]
    labels = [f"{V[a]} {V[b]}" for a, b in answers]
    sc.text(110, 34, "retriever  p(z | x)", size=13, fill="text", weight=700)
    for z in range(3):
        sc.rect(30, 60 + z * 40, 0, 26, fill=cols[z], rx=4).to(0.3, 1.2, w=pz[z] * 220, ease="out")
        sc.text(26, 73 + z * 40, f"z{z+1}", size=12, fill=cols[z], anchor="end")
        sc.num(40 + pz[z] * 220, 73 + z * 40, pz[z], dec=1, size=12, fill="text", anchor="start", opacity=0).fade_in(1.0)
    sc.text(500, 34, "p(answer | x) = sum over z of p(z|x) p(answer | x, z)", size=12, fill="text", weight=700)
    base_y = 330
    bw = 70
    for k, (a, b) in enumerate(answers):
        x = 330 + k * 95
        sc.text(x + bw / 2, base_y + 18, labels[k], size=11, fill="muted")
        y_cur = base_y
        for z in range(3):
            contrib = pz[z] * gen[z, 0, a] * gen[z, 1, b]
            h = contrib * 380
            seg = sc.rect(x, y_cur, bw, 0, fill=cols[z], rx=2)
            t0 = 2.0 + z * 2.0
            seg.to(t0 + 0.1 * k, t0 + 0.9 + 0.1 * k, h=h, y=y_cur - h, ease="out")
            y_cur -= h
        sc.num(x + bw / 2, y_cur - 14, seq[(a, b)], dec=3, size=12, fill="amber", opacity=0).fade_in(8.2)
    sc.beat(0.0, "The retriever scores 3 passages: z1 is most relevant.")
    sc.beat(2.0, "Passage z1's generator strongly prefers 'Paris France', weighted by 0.6...")
    sc.beat(4.0, "...z2 prefers 'Lyon France', weighted by 0.3...")
    sc.beat(6.0, "...and the uninformative z3 adds a thin layer everywhere.")
    sc.beat(8.2, "The stacks are the final probabilities: retrieval uncertainty is averaged out.")
    return sc

marg_scene()

<motion scene: RAG-Sequence: weight each passage's answer distribution by p(z|x), then add (31 elements, 10s)>

## 5. Training: no passage labels needed

Nobody labels which passage answers which question. RAG trains on (question,
answer) pairs alone, maximising $\log p(y \mid x)$ with the marginalised formulas
above. Gradients reach the **query encoder** through $p_\eta(z \mid x)$. If a
passage helps generate the right answer, its score goes up. The paper keeps the
document encoder and its index **fixed**. Re-encoding 21M passages after every
update would be too expensive, and fine-tuning the query encoder alone was
enough.

## 6. Hot-swapping the index

Because knowledge lives in the index rather than the weights, you can change
what the model knows by changing the documents. The paper demonstrated this by
swapping a 2016 Wikipedia index for a 2018 one: questions about world leaders
were then answered according to the newer index, with no retraining. The
cell below does the same thing at small scale.

In [8]:
banner("SWAP THE INDEX, KEEP THE MODEL")
query = "Which model uses a selective state space instead of attention?"
for label, extra in [("original index", None),
                     ("index + one new passage", ("mamba", "The Mamba architecture replaces attention entirely "
                                                           "with a selective state space model."))]:
    d2 = dict(docs)
    if extra:
        d2[extra[0]] = extra[1]
    i2 = list(d2)
    v2 = TfidfVectorizer(stop_words="english").fit([d2[i] for i in i2])
    s2 = (v2.transform([d2[i] for i in i2]) @ v2.transform([query]).T).toarray().ravel()
    top = np.argsort(-s2)[:3]
    print(f"  {label:<24} top-3: " + ", ".join(f"{i2[j]} ({s2[j]:.3f})" for j in top))
print("\n  Before the swap nothing in the index answers the question, yet retrieval still")
print("  returns 3 passages: 'python' ranks first only because it shares the word 'uses'.")
print("  Top-k always returns k results, relevant or not. After the swap the new passage")
print("  ranks first, and neither the retriever nor the generator was retrained.")

  SWAP THE INDEX, KEEP THE MODEL
  original index           top-3: python (0.267), transformer (0.176), rope (0.167)
  index + one new passage  top-3: mamba (0.683), python (0.149), transformer (0.082)

  Before the swap nothing in the index answers the question, yet retrieval still
  returns 3 passages: 'python' ranks first only because it shares the word 'uses'.
  Top-k always returns k results, relevant or not. After the swap the new passage
  ranks first, and neither the retriever nor the generator was retrained.


## Summary

* RAG = a dense retriever (DPR) + a seq2seq generator (BART), trained end to end
  on question–answer pairs, with the retrieved passage as a latent variable.
* Retrieval is maximum inner product search over precomputed passage
  embeddings. The TF-IDF version here ranked the right passage first whenever
  the words overlapped, and scored exactly zero on paraphrases. That gap is what
  learned dense retrievers close.
* **RAG-Sequence** sums over passages outside the product over tokens, and
  **RAG-Token** inside it. Both are proper distributions, checked by full
  enumeration, and they rank answers differently.
* Knowledge in an index can be updated, inspected and cited. Swapping the index
  changes the answers without touching the model.
* Today's "RAG" usually means the inference-time pattern: retrieve, paste into
  the prompt, generate. It often skips the joint training. The ideas of
  retrieval quality capping answer quality and recall@k as the first metric
  carry over unchanged.